# PhishGuard: Phishing URL Detection Web Application

**Objective:** Build a beginner-friendly web app that checks a URL for common structural phishing warning signs and displays a risk indicator.

**How it works:** The app uses transparent heuristic rules. It does not visit the submitted URL, follow redirects, or submit the URL to an external service.

> **Important limitation:** This is an educational heuristic checker, not a definitive phishing classifier. A low score does not prove a URL is safe. The UCI dataset linked in the README is a possible dataset for a future machine-learning extension; this notebook does not train a model on it.

## Setup
1. Install dependencies using `pip install -r requirements.txt`.
2. Run the application cell below.
3. Open `http://127.0.0.1:5000` in your browser.
4. Stop the server with **Interrupt Kernel** in Jupyter.


In [ ]:
from flask import Flask, request, render_template_string
from urllib.parse import urlparse
import ipaddress
import re

app = Flask(__name__)

SUSPICIOUS_KEYWORDS = {
    "login", "verify", "verification", "update", "secure", "account",
    "password", "signin", "banking", "wallet", "confirm", "recover",
    "payment", "gift", "free", "bonus", "support"
}
URL_SHORTENERS = {
    "bit.ly", "tinyurl.com", "t.co", "goo.gl", "is.gd", "buff.ly",
    "ow.ly", "shorturl.at", "cutt.ly", "rebrand.ly"
}
SUSPICIOUS_TLDS = {"zip", "mov", "click", "country", "top", "gq", "tk", "work", "support"}

def inspect_url(raw_url: str) -> dict:
    """Check URL text locally; never visit the submitted destination."""
    original = (raw_url or "").strip()
    if not original:
        return {"valid": False, "error": "Enter a URL to analyze."}

    candidate = original if "://" in original else "http://" + original
    try:
        parsed = urlparse(candidate)
        hostname = parsed.hostname
        _ = parsed.port  # validates malformed port values
    except ValueError:
        parsed, hostname = None, None

    if not parsed or not hostname or "." not in hostname:
        return {"valid": False, "error": "Enter a valid web URL with a domain, such as example.com."}

    host = hostname.lower().rstrip(".")
    reasons, score = [], 0

    def flag(points, message):
        nonlocal score
        score += points
        reasons.append(message)

    if len(original) > 100:
        flag(10, "The URL is unusually long.")
    if "@" in original:
        flag(20, "The URL contains '@', which can disguise the apparent destination.")
    if re.search(r"%[0-9a-fA-F]{2}", original):
        flag(5, "The URL contains percent-encoded characters; inspect it carefully.")
    if host.startswith("xn--") or ".xn--" in host:
        flag(15, "The hostname contains punycode, which can appear in look-alike domains.")
    if "-" in host:
        flag(5, "The hostname contains hyphens.")
    if host.count(".") >= 3:
        flag(10, "The hostname has many subdomain levels.")
    if any(word in original.lower() for word in SUSPICIOUS_KEYWORDS):
        flag(10, "The URL contains a security-sensitive or urgency-related keyword.")
    if host in URL_SHORTENERS:
        flag(15, "This is a URL-shortening service, so the final destination is not visible in the URL.")
    if parsed.scheme.lower() != "https":
        flag(10, "The URL does not use HTTPS. This alone does not prove it is malicious.")
    try:
        ipaddress.ip_address(host)
        flag(20, "The hostname is an IP address rather than a conventional domain.")
    except ValueError:
        pass
    tld = host.rsplit(".", 1)[-1]
    if tld in SUSPICIOUS_TLDS:
        flag(10, f"The .{tld} top-level domain is included in this demo's caution list; this alone does not prove maliciousness.")

    score = min(score, 100)
    if score >= 40:
        verdict, level, css_class = "Potentially suspicious", "High", "danger"
        summary = "Several warning signs were found. Verify the destination independently and do not enter credentials."
    elif score >= 20:
        verdict, level, css_class = "Use caution", "Medium", "warning"
        summary = "Some warning signs were found. Check the domain through a trusted source before interacting."
    else:
        verdict, level, css_class = "No major structural warning signs", "Low", "safe"
        summary = "This basic check found few warning signs. It cannot guarantee that the URL is safe."

    if not reasons:
        reasons.append("No warning signs covered by this basic rule set were detected.")

    return {
        "valid": True, "url": original, "hostname": host, "score": score,
        "verdict": verdict, "level": level, "css_class": css_class,
        "summary": summary, "reasons": reasons
    }

PAGE = r"""
<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>PhishGuard | URL Safety Checker</title>
<style>
:root{color-scheme:dark;--bg:#0a1020;--panel:#111b30;--line:#293852;--text:#eef4ff;--muted:#a8b6ce;--accent:#79e0c3}
*{box-sizing:border-box}body{margin:0;background:radial-gradient(ellipse at top right,#17284a 0,transparent 42%),var(--bg);color:var(--text);font:16px/1.55 system-ui,Segoe UI,sans-serif}
header,main{max-width:1000px;margin:auto;padding:24px}header{display:flex;justify-content:space-between;align-items:center}.brand{font-size:1.2rem;font-weight:800}.tag,.eyebrow{color:var(--accent);font-size:.75rem;font-weight:800;letter-spacing:.12em}
.hero{padding:35px 0 25px;max-width:780px}h1{font-size:clamp(2.4rem,6vw,4.5rem);line-height:1.04;letter-spacing:-.05em;margin:16px 0}h1 span{color:var(--accent)}p{color:var(--muted)}.scanner,.result,.card{background:var(--panel);border:1px solid var(--line);border-radius:16px;padding:22px;margin-top:22px}
label{display:block;font-weight:700;margin-bottom:9px}.row{display:flex;gap:10px}input{flex:1;min-width:0;background:#091225;color:var(--text);border:1px solid #34435e;border-radius:9px;padding:14px;font:inherit}button{background:var(--accent);color:#06221d;border:0;border-radius:9px;padding:13px 18px;font-weight:800;cursor:pointer}.hint{font-size:.85rem}
.result.danger{border-color:#a74d60}.result.warning{border-color:#92743b}.result.safe{border-color:#2b7665}.head{display:flex;justify-content:space-between;gap:15px}.score{min-width:100px;text-align:center;background:#091225;padding:10px;border-radius:12px}.score strong{font-size:2rem}.meter{height:8px;background:#293852;border-radius:10px;overflow:hidden;margin:15px 0}.meter div{height:100%;background:var(--accent)}.danger .meter div{background:#ff7287}.warning .meter div{background:#f5c56b}.detailgrid,.cards{display:grid;grid-template-columns:repeat(2,1fr);gap:12px}.detail{background:#091225;padding:12px;border-radius:9px;overflow-wrap:anywhere}.detail span{display:block;color:var(--muted);font-size:.82rem}.signals li{padding:4px 0}.cards{grid-template-columns:repeat(3,1fr);margin-top:30px}.card{margin:0}.card h3{margin-top:8px}footer{border-top:1px solid var(--line);margin-top:30px;padding-top:15px;font-size:.9rem}
@media(max-width:650px){header,main{padding:18px}.row,.head{flex-direction:column}.cards,.detailgrid{grid-template-columns:1fr}button{width:100%}}
</style></head>
<body>
<header><div class="brand">🛡️ PhishGuard</div><div class="tag">URL SAFETY CHECKER</div></header>
<main>
<section class="hero"><div class="eyebrow">CYBERSECURITY • AWARENESS TOOL</div>
<h1>Spot suspicious links<br><span>before you click.</span></h1>
<p>Analyze a URL's structure for common phishing warning signs. This demo checks URL text locally and does not visit the link.</p>
<form class="scanner" method="post"><label for="url">Website URL</label><div class="row">
<input id="url" name="url" placeholder="https://example.com/login" value="{{ submitted_url }}" required>
<button type="submit">Analyze URL →</button></div><p class="hint">Paste a URL for analysis; do not open a suspicious link to test it.</p></form></section>
{% if result %}
{% if result.valid %}
<section class="result {{ result.css_class }}" aria-live="polite"><div class="head"><div><div class="eyebrow">ANALYSIS RESULT</div><h2>{{ result.verdict }}</h2><p style="overflow-wrap:anywhere">{{ result.url }}</p></div><div class="score"><strong>{{ result.score }}</strong>/100<br><small>Risk indicator</small></div></div>
<div class="meter"><div style="width:{{ result.score }}%"></div></div><p>{{ result.summary }}</p>
<div class="detailgrid"><div class="detail"><span>Risk level</span><strong>{{ result.level }}</strong></div><div class="detail"><span>Hostname</span><strong>{{ result.hostname }}</strong></div></div>
<h3>Signals detected</h3><ul class="signals">{% for reason in result.reasons %}<li>{{ reason }}</li>{% endfor %}</ul></section>
{% else %}<section class="result warning"><h2>Could not analyze this URL</h2><p>{{ result.error }}</p></section>{% endif %}{% endif %}
<section class="cards"><article class="card"><h3>🔎 Structural checks</h3><p>Looks for unusual URL length, IP hosts, misleading characters, and selected patterns.</p></article>
<article class="card"><h3>🔒 Privacy-minded</h3><p>The app does not fetch the destination or send the submitted URL to an external service.</p></article>
<article class="card"><h3>⚠️ Not a guarantee</h3><p>A low score does not prove safety. Sophisticated phishing links may pass.</p></article></section>
<footer><p><strong>Educational project:</strong> use this as a first-pass awareness aid, not as a replacement for a reputable security service.</p></footer>
</main></body></html>
"""

@app.route("/", methods=["GET", "POST"])
def index():
    submitted_url = ""
    result = None
    if request.method == "POST":
        submitted_url = request.form.get("url", "")
        result = inspect_url(submitted_url)
    return render_template_string(PAGE, result=result, submitted_url=submitted_url)

if __name__ == "__main__":
    # Local-only development server. Do not expose it directly to the public internet.
    app.run(host="127.0.0.1", port=5000, debug=False)